# MMHC Quickstart

This notebook demonstrates the Max-Min Hill-Climbing (MMHC) algorithm for learning Bayesian network structure from data.

We'll use the two built-in synthetic datasets:
- **Student network** — 5 variables modelling grades, intelligence, difficulty, SAT scores, and recommendation letters
- **Rainy network** — 3 variables modelling rain, sprinklers, and wet grass

In [ ]:
from mmhc import MMHC, MMHCConfig, mmhc, make_student, make_rainy, plot_dag
import numpy as np

## 1. Student Network

The ground-truth structure is:
```
difficulty ──> grade
intelligence ──> grade
intelligence ──> SAT
grade ──> letter
```

In [ ]:
# Generate 5000 samples from the student network
student_data = make_student(5000, random_state=42)
student_data.head(10)

In [ ]:
student_data.describe()

In [ ]:
# Run MMHC with default config
result = mmhc(student_data, config=MMHCConfig(random_seed=42))

print(f"BDeu score: {result.score:.2f}")
print(f"Converged: {result.converged} in {result.n_iterations} iterations")
print(f"\nRecovered edges:")
for i in range(len(result.column_names)):
    for j in range(len(result.column_names)):
        if result.adjacency_matrix[i, j] == 1:
            print(f"  {result.column_names[i]} -> {result.column_names[j]}")

In [ ]:
# Visualize the learned DAG
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 7))
plot_dag(result.adjacency_matrix, result.column_names, ax=ax, title="Student Network (learned)")
plt.tight_layout()
plt.show()

In [ ]:
# Interactive visualization with pyvis
from mmhc.graph_utils import adjacency_to_networkx
from pyvis.network import Network

def show_interactive(adj, labels, height="400px"):
    g = adjacency_to_networkx(adj, labels)
    net = Network(height=height, width="100%", directed=True, notebook=True, cdn_resources="in_line")
    net.from_nx(g)
    net.set_options('{"physics": {"solver": "forceAtlas2Based"}}')
    return net.show("graph.html")

show_interactive(result.adjacency_matrix, result.column_names)

## 2. Rainy Network

The ground-truth structure is:
```
rain ──> sprinkler
rain ──> grassWet
sprinkler ──> grassWet
```

In [ ]:
rainy_data = make_rainy(5000, random_state=42)
rainy_data.head(10)

In [ ]:
result_rainy = mmhc(rainy_data, config=MMHCConfig(random_seed=42))

print(f"BDeu score: {result_rainy.score:.2f}")
print(f"Converged: {result_rainy.converged} in {result_rainy.n_iterations} iterations")
print(f"\nRecovered edges:")
for i in range(len(result_rainy.column_names)):
    for j in range(len(result_rainy.column_names)):
        if result_rainy.adjacency_matrix[i, j] == 1:
            print(f"  {result_rainy.column_names[i]} -> {result_rainy.column_names[j]}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_dag(result_rainy.adjacency_matrix, result_rainy.column_names, ax=ax, title="Rainy Network (learned)")
plt.tight_layout()
plt.show()

## 3. Adjacency Matrix

The adjacency matrix is a NumPy array where `adj[i, j] = 1` means there is a directed edge from variable `i` to variable `j`.

In [ ]:
import pandas as pd

adj_df = pd.DataFrame(
    result.adjacency_matrix,
    index=result.column_names,
    columns=result.column_names,
)
adj_df.style.map(lambda v: "background-color: #90EE90" if v == 1 else "")

## 4. Using the Class API

The `MMHC` class provides a scikit-learn-style interface with `fit()` and `fit_predict()` methods.

In [ ]:
model = MMHC(config=MMHCConfig(alpha=0.05, eta=1.0, random_seed=42))
result = model.fit(student_data)

# Access via properties
print("Score:", model.score_)
print("Adjacency matrix shape:", model.adjacency_matrix_.shape)
print("Result object:", model.result)